# 01 — Single Product Harness

This notebook is the **single-product gateway** for the Product Evidence Harness. It proves one product end to end and shows the reviewer-facing artifacts, including the browser-visible product-content gate.

```mermaid
flowchart LR
    A[Product Input] --> B[Search]
    B --> C[Candidate Tournament]
    C --> D[Champion Candidate]
    D --> E[Browser-visible Verification]
    E --> F[Production URL Gate]
    F --> G[Concise Review Packet]
    G --> H[Product Coding Input]
```

## Default row packet

```text
output/<row_id>/
├── final_row.csv
├── review_summary.md
├── review_decision.json
├── candidate_decisions.csv
├── browser_visible_verdicts.json
├── browser_visible/
└── product_coding_input.json
```

A URL is not accepted just because it opens. It must also display the intended product in the browser-visible page.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

for path in (PROJECT_ROOT, SRC_PATH):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

PROJECT_ROOT


In [ ]:
from product_evidence_harness import (
    HarnessConfig,
    ProductEvidenceHarness,
    ProductQuery,
    SerpAPIConfig,
    configure_logging,
)

configure_logging("INFO")
print("import ok")


## 1. Product input

Required: `main_text`, `country_code`. Optional: `row_id`, `ean`, `retailer_name`, `language_code`, `region`. Keep EAN/GTIN as text.


In [ ]:
product = ProductQuery(
    row_id="demo-single-001",
    main_text="PUT PRODUCT TEXT HERE",
    country_code="CZ",
    ean="",
    retailer_name="",
)

product


## 2. Configuration

Browser-visible verification is enabled by default. Optional LLM/vision verification is controlled by `PRODUCT_HARNESS_BROWSER_VISIBLE_LLM`.


In [ ]:
config = HarnessConfig.from_env(PROJECT_ROOT / ".env")
tournament = getattr(config, "tournament", None)

pd.Series({
    "output_dir": str(getattr(config, "output_dir", "output")),
    "tournament_enabled": getattr(tournament, "enabled", None),
    "max_serp_credits": getattr(tournament, "max_serp_credits", None),
    "write_review_pack": getattr(config, "write_review_pack", None),
    "browser_visible_verify": getattr(config, "enable_browser_visible_verification", None),
    "require_browser_visible_product_content": getattr(config, "require_browser_visible_product_content", None),
    "browser_visible_capture": getattr(config, "browser_visible_capture_enabled", None),
    "browser_visible_llm": getattr(config, "browser_visible_llm_enabled", None),
    "browser_visible_top_k": getattr(config, "browser_visible_top_k", None),
}).to_frame("value")


## 3. Run the harness

This executes search, tournament, scrape/identity checks, browser-visible verification, champion confirmation, and production gating.


In [ ]:
serp_config = SerpAPIConfig.from_env(
    country_code=product.country_code,
    language_code=product.language_code or "en",
    env_file=PROJECT_ROOT / ".env",
)

harness = ProductEvidenceHarness(serp_config=serp_config, config=config)
trace = harness.run(product, return_trace=True)
match = trace.best_match
match


## 4. Decision snapshot

Automated handoff requires production readiness, browser-visible product match, champion confirmation, and `needs_review=false`.


In [ ]:
tournament_result = getattr(trace.state, "tournament_result", None)
confirmation = getattr(tournament_result, "champion_confirmation", None) if tournament_result else None
production = harness.production_gate.assess_url_in_state(trace.state, match.product_url or "")

decision_summary = {
    "product_url": getattr(match, "product_url", None),
    "best_available_url": getattr(match, "best_available_url", None),
    "verified_exact_url": getattr(match, "verified_exact_url", None),
    "confidence": getattr(match, "confidence", None),
    "needs_review": getattr(match, "needs_review", None),
    "url_decision_status": getattr(match, "url_decision_status", None),
    "production_url_ready": getattr(production, "production_ready", False) if production else False,
    "production_url_status": getattr(production, "status", "NO_PRODUCTION_ASSESSMENT") if production else "NO_PRODUCTION_ASSESSMENT",
    "browser_openable": getattr(production, "browser_openable", False) if production else False,
    "user_visible_product_match": getattr(production, "user_visible_product_match", False) if production else False,
    "user_visible_status": getattr(production, "user_visible_status", "NO_VISIBLE_ASSESSMENT") if production else "NO_VISIBLE_ASSESSMENT",
    "user_visible_page_type": getattr(production, "user_visible_page_type", "unknown") if production else "unknown",
    "user_visible_confidence": getattr(production, "user_visible_confidence", 0.0) if production else 0.0,
    "highly_scrapable": getattr(production, "highly_scrapable", False) if production else False,
    "exact_product_url_match": getattr(production, "exact_product_match", False) if production else False,
    "champion_confirmation_passed": getattr(confirmation, "passed", False) if confirmation else False,
    "champion_confirmation_status": getattr(confirmation, "status", "NO_CONFIRMATION") if confirmation else "NO_CONFIRMATION",
    "tournament_champion_url": getattr(tournament_result, "champion_url", None) if tournament_result else None,
}
pd.Series(decision_summary).to_frame("value")


In [ ]:
handoff_ready = bool(
    production
    and getattr(production, "production_ready", False)
    and getattr(production, "status", "") == "PRODUCTION_READY_EXACT_SCRAPABLE_BROWSER_URL"
    and getattr(production, "user_visible_product_match", False)
    and getattr(production, "user_visible_status", "") == "USER_VISIBLE_PRODUCT_PAGE_CONFIRMED"
    and confirmation
    and getattr(confirmation, "passed", False)
    and getattr(confirmation, "success_count", 0) >= getattr(confirmation, "required_successes", 3)
    and not getattr(match, "needs_review", True)
)

print("HANDOFF_READY_FOR_BROWSER_SCRAPING_AND_PRODUCT_CODING =", handoff_ready)
if not handoff_ready:
    print("This URL may still be useful, but it is review-only until all gates pass.")


## 5. Inspect concise row packet and visible-content artifacts

Open `review_summary.md` first. Use `browser_visible_verdicts.json` when a URL opens but may show the wrong page/content.


In [ ]:
output_dir = Path(getattr(config, "output_dir", PROJECT_ROOT / "output"))
row_dir = output_dir / product.row_id
expected = [
    "final_row.csv", "review_summary.md", "review_decision.json", "candidate_decisions.csv",
    "browser_visible_verdicts.json", "browser_visible", "product_coding_input.json",
]

print("Row artifact directory:", row_dir)
if row_dir.exists():
    existing = {p.name for p in row_dir.iterdir()}
    display(pd.DataFrame({"artifact": expected, "exists": [name in existing for name in expected]}))
else:
    print("Row directory not found. Check PRODUCT_HARNESS_OUTPUT_DIR / config.output_dir.")


## 6. Browser-visible verdicts


In [ ]:
visible_path = row_dir / "browser_visible_verdicts.json"
if visible_path.exists():
    visible = json.loads(visible_path.read_text(encoding="utf-8"))
    rows = []
    for url, verdict in visible.items():
        rows.append({
            "url": url,
            "status": verdict.get("status"),
            "page_type": verdict.get("browser_visible_page_type"),
            "user_visible_product_match": verdict.get("user_visible_product_match"),
            "survives_visible_check": verdict.get("champion_should_survive_visible_check"),
            "confidence": verdict.get("user_visible_content_confidence"),
            "rerouted": verdict.get("rerouted_or_not"),
            "screenshot_path": verdict.get("screenshot_path"),
        })
    display(pd.DataFrame(rows))
else:
    print("browser_visible_verdicts.json not found")


In [ ]:
browser_dir = row_dir / "browser_visible"
if browser_dir.exists():
    files = sorted(p for p in browser_dir.iterdir() if p.is_file())
    display(pd.DataFrame({"file": [p.name for p in files], "path": [str(p) for p in files]}).head(50))
else:
    print("browser_visible folder not found")


## 7. Concise review summary


In [ ]:
summary_path = row_dir / "review_summary.md"
decision_path = row_dir / "review_decision.json"
candidate_path = row_dir / "candidate_decisions.csv"

if summary_path.exists():
    print(summary_path)
    print("\nFirst 1600 characters:\n")
    print(summary_path.read_text(encoding="utf-8")[:1600])
else:
    print("review_summary.md not found")


In [ ]:
if decision_path.exists():
    review_decision = json.loads(decision_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([review_decision.get("decision", {})]).T.rename(columns={0: "value"}))
    display(pd.DataFrame([review_decision.get("checks", {})]).T.rename(columns={0: "value"}))
else:
    print("review_decision.json not found")


In [ ]:
if candidate_path.exists():
    candidates = pd.read_csv(candidate_path, dtype=str)
    review_columns = [
        "rank", "selected", "decision", "url", "confidence", "validation_status",
        "identity_status", "exact_product_check", "country_check", "retailer_check",
        "scrapable", "product_page", "reason",
    ]
    display(candidates[[c for c in review_columns if c in candidates.columns]].head(25))
else:
    print("candidate_decisions.csv not found")
